# fibo-2 — self-hosted example

This notebook runs Bria's **fibo-2** on your own GPU: it asks the Bria Engine for a short-lived package credential, installs **`fibo-2`** from Bria's **`bria-fibo-2`** repository, generates an image from a VGL (the structured JSON caption that fibo-2 reads) and then edits it. The model weights are downloaded from Bria on the first `setup()`.

**Prerequisites**

- A Bria API key in `BRIA_API_TOKEN` — any key of an account that holds a self-hosting license for fibo-2 (https://platform.bria.ai/outpost/licenses?model=fibo-2).
- Python 3.11 or 3.12, and an NVIDIA GPU with at least 40 GB of memory.
- About 26 GB of disk space for the weights.
- Network access to the Bria Engine, AWS CodeArtifact and Bria's weights mirror (CloudFront), and on H100 and H200 GPUs to `huggingface.co`, for the FlashAttention-3 kernel. No Hugging Face account is needed.
- Only for free-form prompts (section 6): an LLM, which writes the VGL of a prompt. Set `GEMINI_API_KEY` for the default (Gemini 3.5 Flash-Lite), or set `FIBO_2_VGL_LLM` (any [litellm](https://docs.litellm.ai/docs/providers) model name) and that provider's credentials, before section 3; see [The LLM for free-form prompts](../docs/README.md#the-llm-for-free-form-prompts). The examples in sections 4 and 5 send a VGL and need no LLM.

## 1. Package credential

Request a short-lived credential for the **`bria-fibo-2`** PyPI repository. The Engine issues it only for a model your account holds a license for; a `403` with `not_entitled` means the account has no license for fibo-2.

In [ ]:
import json
import os
from urllib.error import HTTPError
from urllib.request import Request, urlopen

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN", "").strip()  # a key from a secret file ends in a newline
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN in the environment before running this notebook.")

ENGINE = os.environ.get("BRIA_CLIENT_BASE_URL", "https://engine.prod.bria-api.com")
url = f"{ENGINE}/v2/outpost/access/code_artifact?repository=bria-fibo-2"
request = Request(url, headers={"api_token": BRIA_API_TOKEN})

try:
    with urlopen(request, timeout=60) as response:
        payload = json.loads(response.read().decode("utf-8"))
except HTTPError as exc:
    body = exc.read().decode("utf-8", errors="replace")
    try:
        code = json.loads(body).get("error", {}).get("message")
    except json.JSONDecodeError:
        code = None
    if code in ("not_entitled", "entitlement_revoked"):
        raise RuntimeError(
            "This account has no active license for fibo-2. Get one at https://platform.bria.ai/outpost/licenses?model=fibo-2"
        ) from exc
    raise RuntimeError(f"Package credential request failed with HTTP {exc.code}: {body or exc.reason}") from exc

result = payload.get("result") or {}
CODE_ARTIFACT_PASSWORD = (result.get("authorization_token") or "").strip()
if not CODE_ARTIFACT_PASSWORD:
    raise RuntimeError(f"The response did not include an authorization token: {payload}")
print("Package credential acquired. Expires:", result.get("expiration"))

## 2. Install dependencies

Install **`fibo-2`** from the **`bria-fibo-2`** repository. The same repository carries the build of `diffusers` that fibo-2 needs: its `BriaFibo2Pipeline` is in no `diffusers` release yet. Everything else comes from PyPI.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote

# pip calls os.getcwd(): if Jupyter's working folder was deleted, move to the home folder first.
try:
    Path.cwd()
except FileNotFoundError:
    os.chdir(Path.home())

encoded_password = quote(CODE_ARTIFACT_PASSWORD, safe="")
fibo_2_index = (
    "https://aws:"
    + encoded_password
    + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-fibo-2/simple/"
)


def run_pip_install(install_cmd):
    redacted_cmd = " ".join(part.replace(encoded_password, "<redacted>") for part in install_cmd)
    print("Running:", redacted_cmd, flush=True)
    process = subprocess.Popen(install_cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line.replace(encoded_password, "<redacted>"), end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("pip install failed. See pip output above; authenticated index URLs were redacted.")


run_pip_install(
    [sys.executable, "-m", "pip", "install", "--upgrade", "fibo-2>=0.6.0", "--extra-index-url", fibo_2_index]
)

## 3. Check the license and load the model

`setup()` first checks your license with the Bria Engine, then downloads the fibo-2 weights (about 26 GB) from Bria into `~/.bria/models/fibo-2/<version>/` (set `BRIA_HOME` to move them) and verifies them; later runs reuse them. If the license is missing or has ended, `setup()` raises `LicenseError` with a link to your licenses page and loads nothing. Then it loads the model onto the GPU and runs one small warm-up generation.

If the install cell changed core runtime packages, restart the kernel, run the cell of section 1 again (it sets `BRIA_API_TOKEN`), then run this cell.

In [ ]:
import logging
from pathlib import Path

from common.outpost import LicenseError

from fibo_2 import Fibo2Config, Fibo2Input, Fibo2Pipeline

logging.basicConfig(level=logging.INFO)

outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)

pipeline = Fibo2Pipeline(config=Fibo2Config(api_token=BRIA_API_TOKEN))
try:
    pipeline.setup()
except LicenseError as error:
    raise RuntimeError(f"{error} ({error.code})") from None
print("fibo-2 pipeline setup complete.")

## 4. Generate from a VGL

A VGL is the JSON caption that fibo-2 reads. A request that sends a VGL alone, without a prompt, uses it as it is, so it needs no LLM. `short_description` is the only field that you must give. Each result returns its VGL as `vgl`, with many more fields; send it back to reproduce or adjust the image.

In [ ]:
import json

from IPython.display import display

SAMPLE_VGL = json.dumps(
    {
        "short_description": "A red bicycle leaning against a white brick wall on a sunny morning",
        "style_medium": "photograph",
    }
)

result = pipeline.execute(Fibo2Input(vgl=SAMPLE_VGL, seed=42))
generated_path = outputs_dir / "bicycle.png"
result.image.save(generated_path)
print(f"Generated {result.image.size} in {result.model_seconds}s, saved to {generated_path}")
display(result.image)

## 5. Edit the image with a VGL

An edit sends the images and a VGL with an `edit_instruction`: a command such as "Make the bicycle blue". The other fields describe the image you want back, not the change. This edit needs no LLM either. Without a size, the result takes the shape of the image.

In [ ]:
EDIT_VGL = json.dumps(
    {
        "short_description": "A blue bicycle leaning against a white brick wall on a sunny morning",
        "style_medium": "photograph",
        "edit_instruction": "Make the bicycle blue",
    }
)

edited = pipeline.execute(Fibo2Input(images=[result.image], vgl=EDIT_VGL, seed=42))
edited_path = outputs_dir / "bicycle_blue.png"
edited.image.save(edited_path)
print(f"Edited {edited.image.size} in {edited.model_seconds}s, saved to {edited_path}")
display(edited.image)

## 6. Free-form prompts (needs an LLM)

With `prompt` instead of `vgl`, an LLM first writes the VGL. This example needs the LLM credentials from the prerequisites (a Gemini key, or `FIBO_2_VGL_LLM` and its provider's credentials), set before section 3, because `setup()` checks them. Without an LLM, `setup()` logs a warning, and this cell skips the example.

In [ ]:
if pipeline.vgl_llm is None:
    print(
        "Skipped: free-form prompts need an LLM. Restart the kernel, set GEMINI_API_KEY, or FIBO_2_VGL_LLM and its "
        "provider's credentials (with os.environ, or before you start Jupyter), then run sections 1 and 3 again."
    )
else:
    from_prompt = pipeline.execute(Fibo2Input(prompt="A red bicycle leaning against a white brick wall", seed=42))
    edited_from_prompt = pipeline.execute(
        Fibo2Input(images=[from_prompt.image], prompt="Make the bicycle blue", seed=42)
    )
    from_prompt.image.save(outputs_dir / "bicycle_from_prompt.png")
    edited_from_prompt.image.save(outputs_dir / "bicycle_blue_from_prompt.png")
    print("The VGL that the LLM wrote:", from_prompt.vgl)
    display(from_prompt.image, edited_from_prompt.image)

## 7. Cleanup

Release the GPU memory that the pipeline holds when you are done.

In [ ]:
pipeline.cleanup()
print("Cleanup done.")